# KPI-RAG — Data Pipeline

Builds and validates the modeling arrays (`train_X`, `test_X`) from raw data.

**This notebook owns feature construction.** The modeling notebook (`modeling_pipeline.ipynb`) only ever *loads* the arrays saved at the bottom of this notebook — it never rebuilds features itself. That separation is the fix for the bug this notebook was split out to prevent: a categorical feature block that was reserved (6 zero-initialized columns) but never actually populated, silently, for an unknown number of runs, because the code that computed it lived in a different part of the old monolithic notebook and was never wired into the saved array.

**Column semantics are saved as data, not left as magic numbers.** The last cell in this notebook writes a `feature_schema.json` describing exactly what every column index means. `modeling_pipeline.ipynb` loads `FEAT` from that file instead of hardcoding `576` / `582` / `584` in multiple places.

## Section 1: Environment Setup

In [1]:
# Installs 
%pip install -q shap xgboost lightgbm catboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
%matplotlib inline         
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import ( f1_score, precision_score, recall_score, classification_report, confusion_matrix,
                              precision_recall_curve, average_precision_score )
from sklearn.model_selection import ( StratifiedKFold, StratifiedGroupKFold, RandomizedSearchCV, cross_val_score )
from sklearn.utils import resample
from sklearn.metrics import fbeta_score, make_scorer

import shap
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Paths
DATA_DIR = os.environ.get("KPI_RAG_DATA", "/home/jovyan/kpi_rag/data")
PLOT_DIR = os.path.join(DATA_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# Feature configuration indices (confirmed by exploration)
FEAT = {"A": np.arange(0, 64),
        "B": np.arange(0, 320),
        "C": np.arange(0, 576),
        "C_reduced": np.concatenate([ np.arange(0, 320), np.concatenate([np.arange(320 + (g-1)*16, 320 + g*16)
                                     for g in range(1, 17) if g not in [4, 5, 6, 10, 12, 16] ]) ]) }

assert len(FEAT["A"]) == 64
assert len(FEAT["B"]) == 320
assert len(FEAT["C"]) == 576
assert len(FEAT["C_reduced"]) == 480

# Label constants (confirmed from type_to_int.json)
NORMAL_LABEL = "Normal"
JAMMING_LABEL = "Jamming"
SMALL_CLASSES = {"High Network Congestion (Sudden Spike)", "Resource Allocation Bugs", "Antenna Failure", "Doppler Shift (Severe)"}
NUMERICAL_CHANNELS = ["RSRP", "DL_BLER", "DL_MCS", "UL_BLER", "UL_MCS", "UL_NPRB", "UL_SNR", "TX_Bytes", "RX_Bytes", "Estimated_UL_Buffer",
                       "PRBs_DL_Current", "PRBs_UL_Current", "PRB_Utilization_DL", "PRB_Utilization_UL","UL_NumberOfPackets", "DL_NumberOfPackets"]
assert len(NUMERICAL_CHANNELS) == 16

# Model-lock gate thresholds
GATE_RECALL = 0.50
GATE_F1 = 0.35
SEED = 42

# Helpers 
def save_fig(name):
    path = os.path.join(PLOT_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved: {path}")

def safe_kfold(y, k=5):
    _, counts = np.unique(y, return_counts=True)
    actual = min(k, int(counts.min()))
    if actual < k:
        print(f"  NOTE: folds reduced {k}→{actual} "
              f"(smallest class={counts.min()})")
    return StratifiedKFold(actual, shuffle=True, random_state=SEED)

def safe_group_kfold(y, groups, k=5):
    _, counts = np.unique(y, return_counts=True)
    n_groups = len(np.unique(groups))
    actual = min(k, int(counts.min()), n_groups)
    if actual < k:
        print(f"  NOTE: folds reduced {k}→{actual} (smallest class={counts.min()}, n_groups={n_groups})")
    return StratifiedGroupKFold(actual, shuffle=True, random_state=SEED)

def get_channel_from_feat(fname):
    """Correct channel extraction for multi-word channel names."""
    for ch in NUMERICAL_CHANNELS:
        if fname.startswith(ch):
            return ch
    return fname.split("_")[0]

print("All imports and constants loaded.")
print(f"Feature configs: { {k: len(v) for k,v in FEAT.items()} }")

All imports and constants loaded.
Feature configs: {'A': 64, 'B': 320, 'C': 576, 'C_reduced': 480}


### Feature configuration

`FEAT["C"]` is intentionally **not** hardcoded here anymore. It's computed at the bottom of this notebook from the actual concatenated array shape, then saved to `feature_schema.json`. Configs A/B/C_reduced don't touch the categorical block, so they're safe to define directly.

In [3]:
FEAT = {"A": np.arange(0, 64),
        "B": np.arange(0, 320),
        "C_reduced": np.concatenate([ np.arange(0, 320), np.concatenate([np.arange(320 + (g-1)*16, 320 + g*16)
                                     for g in range(1, 17) if g not in [4, 5, 6, 10, 12, 16] ]) ]) }

assert len(FEAT["A"]) == 64
assert len(FEAT["B"]) == 320
assert len(FEAT["C_reduced"]) == 480
# FEAT["C"] is added below, once the categorical block is built and concatenated.


In [4]:
# Label constants (confirmed from type_to_int.json)
NORMAL_LABEL = "Normal"
JAMMING_LABEL = "Jamming"


## Section 2: Load Raw Data and Existing Numeric Arrays

In [5]:
import os
DATA_DIR = os.environ.get("KPI_RAG_DATA", "/home/jovyan/kpi_rag/data")
for f in sorted(os.listdir(DATA_DIR)):
    print(f)

.ipynb_checkpoints
Images
X_config_A.npy
X_config_A_v2.npy
X_config_B.npy
X_config_B_v2.npy
X_config_C.npy
X_config_C_v2.npy
X_features.npy
X_features_v2.npy
X_test.npy
X_train.npy
algorithm_comparison_other_families.csv
algorithm_comparison_our_results.csv
algorithm_comparison_results.json
algorithm_comparison_sessionsplit.csv
baseline_results.csv
iso_forest.pkl
iso_forest_sessionsplit.pkl
layer2_output.json
layer2_output_sessionsplit.json
layer2_rag_handoff.json
layer2_rag_handoff_anomalies.json
layer2_rag_handoff_sessionsplit.json
model1.pkl
model1_sessionsplit.pkl
model2.pkl
model2_sessionsplit.pkl
model_selection_report.json
model_selection_report_sessionsplit.json
plots
session_id_by_row.npy
shap_importance_A.csv
telecomts_full.parquet
test_X_sessionsplit.npy
test_X_sessionsplit_fixed.npy
test_idx.npy
test_idx_sessionsplit.npy
test_recordings.npy
test_window_stats.npy
test_window_stats_sessionsplit.npy
test_y_sessionsplit.npy
train_X_sessionsplit.npy
train_X_sessionsplit_fixed.np

In [6]:
# Load the numeric portion of the modeling arrays (session-holdout split).
# NOTE: these on-disk arrays currently have a 6-column dead placeholder block
# (cols 576-581, all zero) reserved for categoricals that were never populated.
# We load them here, then REPLACE that dead block with the real categorical
# features built below -- we do not just append on top of the 582 columns.

with open(os.path.join(DATA_DIR, "type_to_int.json")) as f:
    type_to_int = json.load(f)
int_to_type = {v: k for k, v in type_to_int.items()}

train_X_raw = np.load(os.path.join(DATA_DIR, "train_X_sessionsplit.npy"), allow_pickle=True)
test_X_raw  = np.load(os.path.join(DATA_DIR, "test_X_sessionsplit.npy"), allow_pickle=True)
assert train_X_raw.shape[1] == 582, f"Expected 582 cols, got {train_X_raw.shape[1]}"

train_y_str = np.load(os.path.join(DATA_DIR, "train_y_sessionsplit.npy"), allow_pickle=True)
test_y_str  = np.load(os.path.join(DATA_DIR, "test_y_sessionsplit.npy"), allow_pickle=True)
train_y_bin = (train_y_str != NORMAL_LABEL).astype(int)
test_y_bin  = (test_y_str != NORMAL_LABEL).astype(int)

si_train = np.load(os.path.join(DATA_DIR, "train_idx_sessionsplit.npy"))
si_test  = np.load(os.path.join(DATA_DIR, "test_idx_sessionsplit.npy"))

print(f"train_X_raw: {train_X_raw.shape}, test_X_raw: {test_X_raw.shape}")
print(f"train_idx_sessionsplit: {si_train.shape}, test_idx_sessionsplit: {si_test.shape}")


train_X_raw: (25688, 582), test_X_raw: (6312, 582)
train_idx_sessionsplit: (25688,), test_idx_sessionsplit: (6312,)


### ⚠ Split Methodology Note

This pipeline uses a **session-level holdout split** (`*_sessionsplit.npy`), grouping all windows from the same collection session into either train or test — this eliminates the adjacent-window leakage present in the original row-level split.

The plain `train_idx.npy` / `test_idx.npy` files (stratified, window-level, 80/20 of 32,000) are a **leftover from before the session-holdout split was adopted**. They are not used anywhere in this pipeline. Do not load them by mistake — only `*_sessionsplit.npy` files are correct here.

## Section 3: Build Categorical Features from Raw KPIs

In [7]:
import pandas as pd

raw_df = pd.read_parquet("kpi_rag/data/telecomts_full.parquet")
print(raw_df.shape)
print(raw_df.columns.tolist())

(32000, 11)
['start_time', 'end_time', 'sampling_rate', 'KPIs', 'description', 'anomalies', 'statistics', 'labels', 'QnA', 'anomaly_type', 'anomaly_status']


In [8]:
assert len(raw_df) == len(si_train) + len(si_test), (
    f"raw_df has {len(raw_df)} rows but train/test session-split indices sum to "
    f"{len(si_train) + len(si_test)} -- raw_df is not the full unsplit source, "
    "do not index it with these arrays."
)
print("raw_df row count matches the union of session-split indices.")


raw_df row count matches the union of session-split indices.


In [9]:
import numpy as np

def get_kpis(kpi_str):
    """Parse a KPIs field stored as a Python repr() string containing numpy arrays."""
    ns = {
        "array": np.array,
        "nan": np.nan,
        "object": object,
        "float64": np.float64,
        "int64": np.int64,
        "int32": np.int32,
        "float32": np.float32,
        "__builtins__": {},
    }
    return eval(kpi_str, ns)

# Sanity check on one row first
test = get_kpis(raw_df.iloc[0]["KPIs"])
print(type(test))
print(list(test.keys()))
print(test["UL_Protocol"][:5])

<class 'dict'>
['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current', 'PRB_Utilization_DL', 'PRB_Utilization_UL', 'UL_Protocol', 'UL_NumberOfPackets', 'DL_Protocol', 'DL_NumberOfPackets']
['TCP' 'TCP' 'TCP' 'TCP' 'TCP']


Exploration confirmed the dominant-protocol-per-window encoding should use **4 categories per direction** — `TCP`, `UDP`, `None`, and `EMPTY` (empty-string timesteps, which turned out to carry a distinct, non-random pattern rather than being pure missingness) — giving an 8-column block (`UL_*` + `DL_*`).

In [10]:
import numpy as np

PROTOCOL_CATEGORIES = ["TCP", "UDP", "None", "EMPTY"]

def encode_protocol_window(protocol_array):
    """
    Given a 128-length array of per-timestep protocol values for one direction
    (UL or DL), return an 4-element one-hot vector based on the DOMINANT
    (most frequent) protocol across the window.
    """
    # Normalize empty string to explicit "EMPTY" label for clarity
    vals = np.where(protocol_array == '', 'EMPTY', protocol_array)
    vals = pd.Series(vals)
    mode_val = vals.mode()[0]  # dominant protocol in this window

    onehot = np.zeros(len(PROTOCOL_CATEGORIES))
    if mode_val in PROTOCOL_CATEGORIES:
        onehot[PROTOCOL_CATEGORIES.index(mode_val)] = 1.0
    return onehot

def build_categorical_features(raw_df):
    """
    Returns an (n_samples, 8) array: [UL_TCP, UL_UDP, UL_None, UL_EMPTY,
                                        DL_TCP, DL_UDP, DL_None, DL_EMPTY]
    """
    feats = np.zeros((len(raw_df), 8))
    for i in range(len(raw_df)):
        kpis = get_kpis(raw_df.iloc[i]["KPIs"])
        ul_onehot = encode_protocol_window(kpis["UL_Protocol"])
        dl_onehot = encode_protocol_window(kpis["DL_Protocol"])
        feats[i, 0:4] = ul_onehot
        feats[i, 4:8] = dl_onehot
    return feats

cat_col_names = [f"UL_{c}" for c in PROTOCOL_CATEGORIES] + [f"DL_{c}" for c in PROTOCOL_CATEGORIES]

cat_features = build_categorical_features(raw_df)
print(f"Categorical feature block shape: {cat_features.shape}")
print(f"Column names: {cat_col_names}")
print(f"\nColumn-wise activation rate (fraction of windows where this is the dominant protocol):")
for name, rate in zip(cat_col_names, cat_features.mean(axis=0)):
    print(f"  {name}: {rate:.4f}")

Categorical feature block shape: (32000, 8)
Column names: ['UL_TCP', 'UL_UDP', 'UL_None', 'UL_EMPTY', 'DL_TCP', 'DL_UDP', 'DL_None', 'DL_EMPTY']

Column-wise activation rate (fraction of windows where this is the dominant protocol):
  UL_TCP: 0.6623
  UL_UDP: 0.0021
  UL_None: 0.3327
  UL_EMPTY: 0.0029
  DL_TCP: 0.6520
  DL_UDP: 0.0022
  DL_None: 0.3429
  DL_EMPTY: 0.0029


## Section 4: Assemble Final Modeling Arrays

In [11]:
# Split the categorical block with the SAME session-split indices used for train_X/test_X,
# then replace the dead 6-column placeholder (cols 576-581) with the real 8-column block.
cat_train = cat_features[si_train]
cat_test  = cat_features[si_test]
assert cat_train.shape[0] == train_X_raw.shape[0]
assert cat_test.shape[0]  == test_X_raw.shape[0]

train_X = np.concatenate([train_X_raw[:, :576], cat_train], axis=1)
test_X  = np.concatenate([test_X_raw[:, :576],  cat_test],  axis=1)

print(f"train_X: {train_X.shape}  (expected (25688, 584))")
print(f"test_X:  {test_X.shape}  (expected (6312, 584))")


train_X: (25688, 584)  (expected (25688, 584))
test_X:  (6312, 584)  (expected (6312, 584))


## Section 5: Validation Gate

This is the check that would have caught the original bug. The pipeline fails loudly here instead of silently shipping zero-columns downstream.

In [12]:
# --- Validation gate: fail loudly, not silently ---

# 1. No entirely-dead columns anywhere in the final array
dead_cols = np.where(train_X.std(axis=0) == 0)[0]
assert len(dead_cols) == 0, f"Constant/dead columns found at indices: {dead_cols.tolist()}"

# 2. Categorical one-hot blocks each sum to ~1 per row (UL block, then DL block)
cat_block = train_X[:, 576:584]
ul_sums = cat_block[:, 0:4].sum(axis=1)
dl_sums = cat_block[:, 4:8].sum(axis=1)
assert np.allclose(ul_sums, 1.0), "UL categorical block does not sum to 1 per row"
assert np.allclose(dl_sums, 1.0), "DL categorical block does not sum to 1 per row"

# 3. Row counts match labels and indices exactly
assert train_X.shape[0] == len(train_y_bin) == len(si_train)
assert test_X.shape[0]  == len(test_y_bin)  == len(si_test)

# 4. All 12 classes present in test set
missing = set(type_to_int.keys()) - set(test_y_str)
assert len(missing) == 0, f"Missing in test: {missing}"

print("All validation checks passed.")
print(f"Categorical activation rates (train):")
cat_col_names_check = ["UL_TCP","UL_UDP","UL_None","UL_EMPTY","DL_TCP","DL_UDP","DL_None","DL_EMPTY"]
for name, rate in zip(cat_col_names_check, cat_block.mean(axis=0)):
    print(f"  {name}: {rate:.4f}")


All validation checks passed.
Categorical activation rates (train):
  UL_TCP: 0.8158
  UL_UDP: 0.0004
  UL_None: 0.1802
  UL_EMPTY: 0.0036
  DL_TCP: 0.8028
  DL_UDP: 0.0005
  DL_None: 0.1931
  DL_EMPTY: 0.0036


## Section 6: Save Final Arrays and Column Schema

In [13]:
FEAT_C = np.arange(0, 584)

np.save(os.path.join(DATA_DIR, "train_X_final.npy"), train_X)
np.save(os.path.join(DATA_DIR, "test_X_final.npy"), test_X)

feature_schema = {
    "n_features": 584,
    "numeric_channels": 576,
    "categorical_features": 8,
    "categorical_column_names": cat_col_names_check,
    "categorical_column_range": [576, 584],
    "protocol_categories": ["TCP", "UDP", "None", "EMPTY"],
    "configs": {
        "A": [0, 64],
        "B": [0, 320],
        "C": [0, 584],
        "C_reduced_indices": FEAT["C_reduced"].tolist(),
    },
    "source_files": {
        "train_X": "train_X_final.npy",
        "test_X": "test_X_final.npy",
        "train_y": "train_y_sessionsplit.npy",
        "test_y": "test_y_sessionsplit.npy",
        "train_idx": "train_idx_sessionsplit.npy",
        "test_idx": "test_idx_sessionsplit.npy",
    },
    "notes": (
        "Config C is 584 dims (576 numeric + 8 categorical incl. EMPTY protocol state). "
        "This supersedes all prior 576-dim Config C results computed before the dead "
        "zero-column bug was fixed -- those must be rerun."
    ),
}

with open(os.path.join(DATA_DIR, "feature_schema.json"), "w") as f:
    json.dump(feature_schema, f, indent=2)

print("Saved train_X_final.npy, test_X_final.npy, feature_schema.json")
print(f"FEAT[\"C\"] = np.arange(0, 584)  -- {len(FEAT_C)} features")


Saved train_X_final.npy, test_X_final.npy, feature_schema.json
FEAT["C"] = np.arange(0, 584)  -- 584 features
